In [0]:
%sql
CREATE OR REPLACE TABLE synchrony.analytics.customer_monthly_sow AS

SELECT

    Customer_ID,

    YEAR(Transaction_Date) AS Calendar_Year,

    MONTH(Transaction_Date) AS Month_Number,

    DATE_TRUNC('MONTH', Transaction_Date) AS Month,

    CASE
        WHEN MONTH(Transaction_Date) >= 8
            THEN YEAR(Transaction_Date) + 1
        ELSE YEAR(Transaction_Date)
    END AS Fiscal_Year,

    CASE
        WHEN MONTH(Transaction_Date) IN (8,9,10)
            THEN 'Q1'

        WHEN MONTH(Transaction_Date) IN (11,12,1)
            THEN 'Q2'

        WHEN MONTH(Transaction_Date) IN (2,3,4)
            THEN 'Q3'

        WHEN MONTH(Transaction_Date) IN (5,6,7)
            THEN 'Q4'
    END AS Fiscal_Quarter,

    Membership_Type,

    SUM(Net_Amount) AS Total_MetroMart_Spend,

    SUM(
        CASE
            WHEN Payment_Code = 3
                THEN Net_Amount
            ELSE 0
        END
    ) AS HSIC_Spend,

    SUM(
        CASE
            WHEN Payment_Code <> 3
                THEN Net_Amount
            ELSE 0
        END
    ) AS Other_Payment_Spend,

    SUM(
        CASE
            WHEN Payment_Method = 'Other Bank Credit Card'
                THEN Net_Amount
            ELSE 0
        END
    ) AS Competitor_Card_Spend,

    SUM(
        CASE
            WHEN Payment_Method = 'Debit Card'
                THEN Net_Amount
            ELSE 0
        END
    ) AS Debit_Card_Spend,

    SUM(
        CASE
            WHEN Payment_Method = 'Cash/UPI'
                THEN Net_Amount
            ELSE 0
        END
    ) AS Cash_UPI_Spend,

    SUM(
        CASE
            WHEN Payment_Method = 'MetroMart Wallet'
                THEN Net_Amount
            ELSE 0
        END
    ) AS Wallet_Spend,

    SUM(
        CASE
            WHEN Payment_Code = 3
                THEN Net_Transactions
            ELSE 0
        END
    ) AS HSIC_Transactions,

    SUM(Net_Transactions) AS Total_Transactions

FROM synchrony.analytics.active_transactions

GROUP BY

    Customer_ID,
    YEAR(Transaction_Date),
    MONTH(Transaction_Date),
    DATE_TRUNC('MONTH', Transaction_Date),

    CASE
        WHEN MONTH(Transaction_Date) >= 8
            THEN YEAR(Transaction_Date) + 1
        ELSE YEAR(Transaction_Date)
    END,

    CASE
        WHEN MONTH(Transaction_Date) IN (8,9,10) THEN 'Q1'
        WHEN MONTH(Transaction_Date) IN (11,12,1) THEN 'Q2'
        WHEN MONTH(Transaction_Date) IN (2,3,4) THEN 'Q3'
        WHEN MONTH(Transaction_Date) IN (5,6,7) THEN 'Q4'
    END,

    Membership_Type;